<a href="https://colab.research.google.com/github/aliraza-chaudhary/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aliraza-chaudhary/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Unit of analysis: One row represents one webpage for one client on one reporting date.

Time window: I will use March 2026 as the development month. I will treat June 2026 as a sealed test period and will not use it for feature or label development.

In [49]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Features

For the content-refresh lane, I will use these five features:

- `gsc_impressions` — search visibility available before the decision.
- `gsc_clicks` — search clicks available before the decision.
- `gsc_avg_position` — average search position available before the decision.
- `ga4_sessions` — website sessions available before the decision.
- `scroll_events` — observed engagement available before the decision.

### Label

The warehouse table does not contain the final `trend_direction` label. I will construct the outcome later from future search-performance changes rather than using a label-derived field from the same observation.

### Context

`content_hash_id`, `client_hash_id`, `report_date`, and `month` identify the webpage, client, and time period.

### Excluded

I will exclude identifiers such as `content_hash_id` and `client_hash_id` from predictive features because they identify entities rather than describe their observable characteristics. I will also exclude fields derived from the future outcome to avoid leakage.

In [50]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [51]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [52]:
# Connect to the Hugging Face warehouse
from google.colab import userdata
import duckdb

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf_secret "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

base = "hf://datasets/FlyRank/internship-warehouse"

print("QUERY 1 — Grain check")

grain = con.sql(f"""
    SELECT
        COUNT(*) AS rows,
        COUNT(DISTINCT
            content_hash_id || '-' ||
            client_hash_id || '-' ||
            CAST(report_date AS VARCHAR)
        ) AS unique_content_client_date_rows
    FROM read_parquet(
        '{base}/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
""").df()

display(grain)


# --------------------------------------------------
# QUERY 2 — Row count and date span
# --------------------------------------------------
print("QUERY 2 — March 2026 count and date span")

count_dates = con.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date
    FROM read_parquet(
        '{base}/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
""").df()

display(count_dates)

# --------------------------------------------------
# QUERY 3 — Availability check using IS TRUE
# --------------------------------------------------
print("QUERY 3 — GSC Availability")

availability = con.sql(f"""
    SELECT
        COUNT(*) AS rows_before_filter,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS rows_after_filter
    FROM read_parquet(
        '{base}/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
""").df()

display(availability)

QUERY 1 — Grain check


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rows,unique_content_client_date_rows
0,9841378,9841378


QUERY 2 — March 2026 count and date span


,row_count,first_date,last_date
0,9841378,2026-03-01,2026-03-31


QUERY 3 — GSC Availability


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rows_before_filter,rows_after_filter
0,9841378,3611061


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Data limits

This data can show observed and directional relationships between webpage characteristics and search performance, but it cannot prove that refreshing a page will cause traffic or ranking improvements.

The history may be unbalanced across clients and pages, so results may not represent every client equally. The daily performance table also contains multiple observations for the same webpage across dates, so these rows should not be treated as fully independent.

The available fields also do not contain the final future-outcome label, so the label will need to be constructed from the time series. I will use March 2026 as a development period and treat the final June 2026 month as a sealed test period.

Results will therefore be used as decision support for content-refresh prioritization, not as causal proof.

In [53]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### Five features

For March 2026, I will create a small feature frame using five features:

1. `gsc_impressions` — knowable at the decision moment because it records search impressions available before prioritizing a refresh.
2. `gsc_clicks` — knowable at the decision moment because it records search clicks available before prioritizing a refresh.
3. `gsc_avg_position` — knowable at the decision moment because it describes the page's observed search position before the refresh decision.
4. `ga4_sessions` — knowable at the decision moment because it records observed website sessions before the decision.
5. `scroll_events` — knowable at the decision moment because it records observed engagement before the decision.

In [54]:
features = con.sql(f"""
    SELECT
        content_hash_id,
        client_hash_id,
        report_date,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,
        ga4_sessions,
        scroll_events
    FROM read_parquet(
        '{base}/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
    LIMIT 10
""").df()

display(features)

,content_hash_id,client_hash_id,report_date,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,scroll_events
0,content_b7e512995f79d5a6,client_73cda7b4e4f265ea,2026-03-01,20,0,3.350000,<NA>,<NA>
1,content_05597932fe4da067,client_73cda7b4e4f265ea,2026-03-01,1,0,0.000000,<NA>,<NA>
2,content_7a105f548d9c6916,client_73cda7b4e4f265ea,2026-03-01,125,1,4.928000,<NA>,<NA>
3,content_905aa32a0230694e,client_73cda7b4e4f265ea,2026-03-01,7,0,4.000000,<NA>,<NA>
4,content_a3ea9792f793ec72,client_73cda7b4e4f265ea,2026-03-01,11,0,2.272727,<NA>,<NA>
5,content_36c36abc7650d7af,client_73cda7b4e4f265ea,2026-03-01,239,1,7.347280,<NA>,<NA>
6,content_a7da352b73b02668,client_73cda7b4e4f265ea,2026-03-01,191,0,7.832461,<NA>,<NA>
7,content_05434271b257bb68,client_73cda7b4e4f265ea,2026-03-01,55,0,3.272727,<NA>,<NA>
8,content_d056587ff7faca0c,client_73cda7b4e4f265ea,2026-03-01,77,0,5.636364,<NA>,<NA>
9,content_bfd1e41c2af250c8,client_73cda7b4e4f265ea,2026-03-01,2,0,4.500000,<NA>,<NA>


### Leakage experiment

To demonstrate leakage, I intentionally create a feature from the future outcome. This feature would not be available when deciding which page to refresh. I expect it to make the quick score unrealistically strong because it contains information derived from the answer.

After observing the inflated score, I remove the leaked feature and keep only features that would be available at the decision moment.

In [55]:
# Leakage experiment
# We use March as the decision period and April as the future outcome period.

query = f"""
WITH march AS (
    SELECT
        content_hash_id,
        client_hash_id,
        AVG(gsc_clicks) AS march_clicks
    FROM read_parquet(
        '{base}/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
    GROUP BY content_hash_id, client_hash_id
),

april AS (
    SELECT
        content_hash_id,
        client_hash_id,
        AVG(gsc_clicks) AS april_clicks
    FROM read_parquet(
        '{base}/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-04'
    GROUP BY content_hash_id, client_hash_id
)

SELECT
    m.content_hash_id,
    m.client_hash_id,
    m.march_clicks,
    a.april_clicks,

    CASE
        WHEN a.april_clicks < m.march_clicks THEN 1
        ELSE 0
    END AS is_declining

FROM march m
INNER JOIN april a
    ON m.content_hash_id = a.content_hash_id
    AND m.client_hash_id = a.client_hash_id
"""

leak_df = con.sql(query).df()

display(leak_df.head())
print("Rows:", len(leak_df))
print("Declining pages:", leak_df["is_declining"].sum())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,client_hash_id,march_clicks,april_clicks,is_declining
0,content_b7e512995f79d5a6,client_73cda7b4e4f265ea,0.064516,0.066667,0
1,content_a7da352b73b02668,client_73cda7b4e4f265ea,0.419355,0.266667,1
2,content_d056587ff7faca0c,client_73cda7b4e4f265ea,0.516129,0.200000,1
3,content_bfd1e41c2af250c8,client_73cda7b4e4f265ea,0.000000,0.000000,0
4,content_2662845f598544ef,client_73cda7b4e4f265ea,0.032258,0.000000,1


Rows: 331436
Declining pages: 45744


### Deliberate leakage

I constructed `is_declining` from the future April performance relative to March. This is the outcome I want to predict, so it would be unavailable at the March decision moment.

For the leakage experiment, I intentionally use this outcome-derived value as a feature. A model using this information can appear unrealistically accurate because the feature directly contains the answer.

I then remove this leaked feature and keep only March features that were available at the decision moment.

In [56]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Keep only complete rows for this demonstration
demo = leak_df.dropna(subset=["march_clicks", "april_clicks", "is_declining"]).copy()

X = demo[["march_clicks"]].copy()
y = demo["is_declining"]

# DELIBERATE LEAK:
# is_declining itself is used as a feature.
X["leaked_outcome"] = y

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = DecisionTreeClassifier(max_depth=2, random_state=42)
model.fit(X_train, y_train)

pred = model.predict(X_test)

leaked_score = accuracy_score(y_test, pred)

print("Accuracy with deliberate leakage:", leaked_score)

Accuracy with deliberate leakage: 1.0


### Removing the leak

The previous score is not an honest estimate because `leaked_outcome` directly contains the answer. I remove it and keep only information that would have been available when the refresh decision was made.

In [57]:
# Remove the leaked feature
X_honest = demo[["march_clicks"]].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X_honest, y, test_size=0.2, random_state=42
)

honest_model = DecisionTreeClassifier(max_depth=2, random_state=42)
honest_model.fit(X_train, y_train)

honest_pred = honest_model.predict(X_test)

honest_score = accuracy_score(y_test, honest_pred)

print("Honest accuracy after removing leakage:", honest_score)

Honest accuracy after removing leakage: 0.9291274438812455


### Leakage conclusion

The deliberately leaked version produced an artificially strong score because the model was given information derived directly from the future outcome. After removing the leaked feature, the honest score is lower and represents a more realistic decision-support result.

The leaked feature will not be included in the final feature set.

### Named limitation

A key limitation is uneven historical coverage across webpages and clients. Some pages may have less complete performance history or unavailable GSC data, which can limit how confidently results generalize across the full dataset.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.